<a href="https://colab.research.google.com/github/hussainturii/Advanced-Computer-Vision---Deep-Learning/blob/main/VGGNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision import models
from torchsummary import summary
import wandb
import os

In [2]:
# =======================
# STEP 0: Initialize wandb
# =======================
wandb.init(project="VGG-flowers-v3", config={
    "epochs": 50, #better training accuracy
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "VGG",
    "pretrained": True,
    "input_size": 224 # must be of this size
})

# Shortcut to config values
config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hussainturi (hussaintoori) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# =======================
# STEP 1: Data Preparation
# =======================

# Transforms for training and validation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)), # in case image is not 224*244
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

train_dir = "/content/drive/MyDrive/flowers/train"
val_dir = "/content/drive/MyDrive/flowers/val"

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

In [4]:
# ===========================
# STEP 2: Load Pretrained Model
# ===========================

model = models.vgg16(pretrained=config.pretrained)

# Freeze feature layers, we only want to use the last layer.
# the other layers work as a blackbox
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
# last layer config:
model.classifier = nn.Sequential(
    nn.Linear(25088, 4096),
    nn.ReLU(inplace=True),
    nn.Dropout(),
    nn.Linear(4096, 4096),
    nn.ReLU(inplace=True),
    nn.Dropout(),
    nn.Linear(4096, 5)  # 5 output classes for flower dataset
)

# Only unfreeze the final classifier layer (index -1)
for param in model.classifier[-1].parameters():
    param.requires_grad = True

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Watch the model's weights and gradients
wandb.watch(model, log="all", log_freq=10)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 91.1MB/s]


In [5]:
# ===================
# STEP 3: Loss & Optimizer
# ===================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

In [6]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0
        running_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            batch_correct = (preds == labels).sum().item()
            train_correct += batch_correct
            train_total += labels.size(0)

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                batch_acc = batch_correct / labels.size(0)
                print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f}, Batch Acc: {batch_acc:.4f}")

        train_acc = train_correct / train_total
        wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
        print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
        print(f"Validation Accuracy: {val_acc:.4f}")


In [ ]:
# ===================
# Train the model
# ===================
train_model(model, criterion, optimizer, train_loader, val_loader, epochs=config.epochs)


Epoch 1/50
------------------------------
[Batch 10/251] Loss: 1.2729, Batch Acc: 0.4375
[Batch 20/251] Loss: 0.9865, Batch Acc: 0.6250
[Batch 30/251] Loss: 1.3926, Batch Acc: 0.5625
[Batch 40/251] Loss: 1.0126, Batch Acc: 0.8125
[Batch 50/251] Loss: 0.9491, Batch Acc: 0.8750
[Batch 60/251] Loss: 1.4132, Batch Acc: 0.6875
[Batch 70/251] Loss: 1.2172, Batch Acc: 0.6250
[Batch 80/251] Loss: 1.3019, Batch Acc: 0.5625
[Batch 90/251] Loss: 1.1304, Batch Acc: 0.7500
[Batch 100/251] Loss: 0.8374, Batch Acc: 0.6250
[Batch 110/251] Loss: 1.0230, Batch Acc: 0.7500
[Batch 120/251] Loss: 0.6645, Batch Acc: 0.8125
[Batch 130/251] Loss: 0.8026, Batch Acc: 0.6875
[Batch 140/251] Loss: 1.0957, Batch Acc: 0.6875
[Batch 150/251] Loss: 0.8846, Batch Acc: 0.7500
[Batch 160/251] Loss: 0.6281, Batch Acc: 0.8750
[Batch 170/251] Loss: 0.5470, Batch Acc: 0.8750
[Batch 180/251] Loss: 0.6071, Batch Acc: 0.8125
[Batch 190/251] Loss: 0.3616, Batch Acc: 0.8750
[Batch 200/251] Loss: 1.1722, Batch Acc: 0.8125
[Batch